# 070 CASE 020 — Wildfire: Kårböle (Ljusdal, Gävleborg)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_020-Wildfire-Karbole.ipynb)

**Purpose:** Map burn severity using Sentinel-2 after the 2018 Kårböle wildfire (Ljusdalsbranden — one of the largest fires in modern Swedish history, ~8,400 ha).

**Partners:**
- **Skogsstyrelsen** — field reference
- **RISE** — ImintEngine
- **MSB** — potential end-user for crisis preparedness

**Licence:** CC0 1.0 notebook.

## 1. Method — dNBR from two SpectralAnalyzer runs

$$\text{NBR} = \frac{\text{B08}-\text{B12}}{\text{B08}+\text{B12}}  \qquad  \text{dNBR} = \text{NBR}_{\text{pre}} - \text{NBR}_{\text{post}}$$

Higher dNBR = more severe burn. USGS classifies dNBR > 0.66 as *high severity*.

`SpectralAnalyzer` is in `ANALYZER_REGISTRY` and exposes the index dict `{NDVI, NDWI, NDBI, EVI, NBR}`. We run it once on the pre-fire date and once on the post-fire date, then compute dNBR manually.

The `prithvi` analyzer (IBM/NASA burn-scars head) would normally provide an AI-segmented burn mask — it requires the full 6-band Prithvi set (B02/B03/B04/B8A/B11/B12). On a pod with only synthetic fallback data it cannot run; the notebook handles that gracefully.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
from imint.engine import run_job
import numpy as np
import matplotlib.pyplot as plt
import folium

def get_outputs(result, name):
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None

AOI = {"west": 15.35, "south": 61.78, "east": 15.75, "north": 62.00}
PRE_FIRE = "2018-06-25"
POST_FIRE = "2018-08-30"

print(f"AOI (Kårböle): {AOI}")
print(f"Pre-fire: {PRE_FIRE}")
print(f"Post-fire: {POST_FIRE}")

## 3. Run analysis — two acquisitions

In [ ]:
executor = LocalExecutor(output_dir="outputs/brand_karbole")

print("--- PRE-FIRE ---")
pre_job = executor.build_job(date=PRE_FIRE, coords=AOI)
pre = run_job(pre_job)

print("--- POST-FIRE ---")
post_job = executor.build_job(date=POST_FIRE, coords=AOI)
post = run_job(post_job)

pre_spec = get_outputs(pre, "spectral")
post_spec = get_outputs(post, "spectral")

if pre_spec is not None and post_spec is not None:
    nbr_pre = pre_spec["indices"]["NBR"]
    nbr_post = post_spec["indices"]["NBR"]
    dnbr = nbr_pre - nbr_post
    print(f"\ndNBR stats:")
    print(f"  mean:  {np.nanmean(dnbr):.4f}")
    print(f"  max:   {np.nanmax(dnbr):.4f}")
    # USGS severity classes
    pct_high = float((dnbr > 0.66).mean() * 100)
    pct_mod = float(((dnbr > 0.27) & (dnbr <= 0.66)).mean() * 100)
    print(f"  high severity (>0.66):     {pct_high:.2f}%")
    print(f"  moderate (0.27..0.66):     {pct_mod:.2f}%")
else:
    print("Could not compute dNBR — spectral analyzer missing in one of the runs.")
    dnbr = None

## 4. Visualize

In [ ]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=11, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite",
).add_to(m)
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#d73a49", weight=2, fill=False,
    popup="Kårböle fire 2018 (Ljusdalsbranden)",
).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(pre_job.rgb)
axes[0, 0].set_title(f"Pre-fire RGB ({PRE_FIRE})")
axes[0, 0].axis("off")

axes[0, 1].imshow(post_job.rgb)
axes[0, 1].set_title(f"Post-fire RGB ({POST_FIRE})")
axes[0, 1].axis("off")

if dnbr is not None:
    im = axes[1, 0].imshow(dnbr, cmap="RdYlGn_r", vmin=-0.5, vmax=1.0)
    axes[1, 0].set_title("dNBR (higher = more severe burn)")
    axes[1, 0].axis("off")
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046)
else:
    axes[1, 0].text(0.5, 0.5, "dNBR unavailable", ha="center", va="center")
    axes[1, 0].axis("off")

prithvi = get_outputs(post, "prithvi")
if prithvi is not None:
    # Prithvi exact output keys vary by task head; pick the first ndarray
    arr_key = next((k for k, v in prithvi.items() if hasattr(v, "shape")), None)
    if arr_key:
        axes[1, 1].imshow(prithvi[arr_key], cmap="hot")
        axes[1, 1].set_title(f"Prithvi-EO: {arr_key}")
    else:
        axes[1, 1].text(0.5, 0.5, "prithvi outputs unavailable", ha="center", va="center")
else:
    axes[1, 1].text(0.5, 0.5, "prithvi requires full band set\n(not available on fallback data)",
                    ha="center", va="center")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Interpretation

**What dNBR shows:**
- Continuous burn-severity scale — USGS thresholds: 0.10 = unburnt, 0.27+ = moderate, 0.66+ = high
- Captures *vegetation loss* via NIR/SWIR contrast — unique to fire vs other land-cover change

**Reference:**
- Skogsstyrelsen's official damage inventory: ~8,400 ha burn area
- AOI here covers the main Enskogen/Kårböle zone

**Full-fidelity run needs:**
- Real Sentinel-2 pre/post-fire data (cloud-free) via DES or CDSE
- Prithvi-EO weights (~1.2 GB) for AI segmentation

### Links
- [`imint/analyzers/spectral.py`](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/spectral.py)
- [Prithvi-EO paper](https://arxiv.org/abs/2310.18660)
- [USGS dNBR severity](https://burnseverity.cr.usgs.gov/)